# 使用秘塔META API获取最新技术资讯

本笔记本演示如何使用秘塔META搜索API，获取感兴趣领域的最新资讯。

**参考API**: https://metaso.cn/api/v1/search

## 步骤1: 导入必要的库

In [2]:
import json
import requests
import os
from dotenv import load_dotenv
from datetime import datetime
import re
from typing import List, Dict, Any

print("✓ 库导入成功")

✓ 库导入成功


## 步骤2: 加载环境变量和配置

从 `.env` 文件中加载API密钥

In [3]:
# 加载环境变量
load_dotenv()

# 获取API凭证
META_API_KEY = os.getenv('META_API_KEY')

# API配置
API_URL = "https://metaso.cn/api/v1/search"

# 验证API密钥是否加载成功
if META_API_KEY:
    print(f"✓ API密钥已加载 (长度: {len(META_API_KEY)} 字符)")
else:
    print("✗ 警告: API密钥未找到，请检查.env文件")

✓ API密钥已加载 (长度: 35 字符)


## 步骤3: 定义感兴趣的技术领域

自定义你想要获取资讯的技术领域

In [4]:
# 定义感兴趣的技术领域
TECH_FIELDS = [
    "自动驾驶",
    "具身智能",
    "大模型",
    "人工智能"
]

# 定义重点关注的主题
FOCUS_TOPICS = [
    "特斯拉FSD",
    "人形机器人",
    # "AI大模型突破",
    # "行业重要动态"
]

# 定义搜索资源范围
# 支持的scope类型: 'webpage' (网页), 'blog' (博客), 'scholar' (学术)
SEARCH_SCOPES = ['webpage', 'blog', 'scholar']

# 定义时间范围过滤
# 可选值: None (不限), 'day' (最近1天), 'week' (最近7天), 'month' (最近30天)
TIME_FILTER = 'day'  # 设置为最近1天

print(f"关注领域: {', '.join(TECH_FIELDS)}")
print(f"重点主题: {', '.join(FOCUS_TOPICS)}")
print(f"搜索范围: {', '.join(SEARCH_SCOPES)}")
print(f"时间过滤: {TIME_FILTER if TIME_FILTER else '不限'}")

关注领域: 自动驾驶, 具身智能, 大模型, 人工智能
重点主题: 特斯拉FSD, 人形机器人
搜索范围: webpage, blog, scholar
时间过滤: day


## 步骤4: 创建META API搜索函数

构建调用META搜索API的核心函数

In [ ]:
def search_meta_api(query: str, api_key: str, size: int = 10, 
                    scope: str = "webpage",
                    time_filter: str = None,
                    include_summary: bool = False, 
                    include_raw_content: bool = False) -> Dict[str, Any]:
    """
    调用META搜索API获取搜索结果
    
    参数:
        query: 搜索查询词
        api_key: API密钥
        size: 返回结果数量 (默认10)
        scope: 搜索范围，可选 'webpage', 'blog', 'scholar'
        time_filter: 时间过滤，可选 'day' (最近1天), 'week' (最近7天), 'month' (最近30天), None (不限)
        include_summary: 是否包含摘要
        include_raw_content: 是否包含原始内容
    
    返回:
        搜索结果字典
    """
    # 构建请求头
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Accept': 'application/json',
        'Content-Type': 'application/json'
    }
    
    # 根据时间过滤构建查询
    final_query = query
    if time_filter:
        from datetime import datetime, timedelta
        
        if time_filter == 'day':
            # 最近1天
            final_query = f"{query} 最近1天"
        elif time_filter == 'week':
            # 最近1周
            final_query = f"{query} 最近一周"
        elif time_filter == 'month':
            # 最近1月
            final_query = f"{query} 最近一月"
    
    # 构建请求体
    body = {
        "q": final_query,
        "scope": scope,
        "includeSummary": include_summary,
        "size": str(size),
        "includeRawContent": include_raw_content,
        "conciseSnippet": False
    }
    
    try:
        scope_name = {'webpage': '网页', 'blog': '博客', 'scholar': '学术'}.get(scope, scope)
        time_desc = {'day': '(最近1天)', 'week': '(最近7天)', 'month': '(最近30天)', None: ''}.get(time_filter, '')
        print(f"正在搜索 [{scope_name}{time_desc}]: {query}...")
        
        print(f"{API_URL=}, json={body}, headers={headers}")
        response = requests.post(API_URL, json=body, headers=headers, timeout=30)
        
        # 检查HTTP状态码
        if response.status_code != 200:
            print(f"✗ API返回错误状态码: {response.status_code}")
            print(f"响应内容: {response.text}")
            return None
        
        print(f"✓ 搜索成功")
        
        # 解析响应
        data = response.json()
        print(data)
        input()
        riase ValueError
        
        # return data
        print(f"{API_URL = }\nbody = {json.dumps(body)} \n headers = {json.dumps(headers)}")
        return {}
            
    except requests.exceptions.Timeout:
        print("✗ API调用超时")
        return None
    except Exception as e:
        print(f"✗ API调用错误: {e}")
        return None

print("✓ META API搜索函数已定义（支持时间过滤和多资源类型）")

✓ META API搜索函数已定义（支持时间过滤和多资源类型）


## 步骤5: 创建多主题搜索函数

为多个技术领域和主题执行搜索

In [6]:
def search_multiple_topics(fields: List[str], focus_topics: List[str], 
                          api_key: str, 
                          scopes: List[str] = ['webpage'],
                          time_filter: str = None,
                          results_per_topic: int = 3) -> Dict[str, Any]:
    """
    对多个技术领域和主题进行搜索，支持多资源类型和时间过滤
    
    参数:
        fields: 技术领域列表
        focus_topics: 重点主题列表
        api_key: API密钥
        scopes: 搜索范围列表，如 ['webpage', 'blog', 'scholar']
        time_filter: 时间过滤，可选 'day', 'week', 'month', None
        results_per_topic: 每个主题返回的结果数
    
    返回:
        汇总的搜索结果
    """
    all_results = {
        "search_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "fields": fields,
        "focus_topics": focus_topics,
        "scopes": scopes,
        "time_filter": time_filter,
        "news_items": [],
        "total_results": 0,
        "stats_by_scope": {}
    }
    
    # 组合搜索查询
    search_queries = []
    
    # 为每个领域创建搜索查询
    for field in fields:
        search_queries.append(field)
    
    # 为每个重点主题创建搜索查询
    for topic in focus_topics:
        search_queries.append(topic)
    
    # 计算总搜索次数
    total_searches = len(search_queries) * len(scopes)
    
    print(f"\n总共需要执行 {total_searches} 次搜索")
    print(f"  - 查询主题: {len(search_queries)} 个")
    print(f"  - 资源类型: {len(scopes)} 种 ({', '.join(scopes)})")
    print(f"  - 时间过滤: {time_filter if time_filter else '不限'}")
    print("\n" + "="*80)
    
    # 执行搜索
    item_id = 1
    search_count = 0
    
    # 为每个scope初始化统计
    for scope in scopes:
        all_results['stats_by_scope'][scope] = 0
    
    for scope in scopes:
        scope_name = {'webpage': '网页', 'blog': '博客', 'scholar': '学术'}.get(scope, scope)
        print(f"\n{'='*80}")
        print(f"正在搜索资源类型: {scope_name}")
        print(f"{'='*80}")
        
        for idx, query in enumerate(search_queries, 1):
            search_count += 1
            print(f"\n[{search_count}/{total_searches}] 主题: {query}")
            
            result = search_meta_api(
                query=query, 
                api_key=api_key, 
                size=results_per_topic,
                scope=scope,
                time_filter=time_filter
            )
            input(f"暂停，按回车继续...")
            
            if result and 'webpages' in result:
                webpages = result['webpages']
                print(f"  找到 {len(webpages)} 条结果")
                
                # 处理每条结果
                for webpage in webpages:
                    news_item = {
                        "id": item_id,
                        "category": query,
                        "title": webpage.get('title', 'N/A'),
                        "content": webpage.get('snippet', 'N/A'),
                        "source_link": webpage.get('link', 'N/A'),
                        "score": webpage.get('score', 'N/A'),
                        "date": webpage.get('date', 'N/A'),
                        "position": webpage.get('position', 0),
                        "query": query,
                        "scope": scope,  # 添加资源类型标记
                        "scope_name": scope_name
                    }
                    all_results['news_items'].append(news_item)
                    item_id += 1
                
                all_results['total_results'] += len(webpages)
                all_results['stats_by_scope'][scope] += len(webpages)
            else:
                print(f"  未找到结果")
            
            print("-"*80)
    
    print(f"\n✓ 搜索完成! 总共获取 {all_results['total_results']} 条资讯")
    print(f"\n按资源类型统计:")
    for scope, count in all_results['stats_by_scope'].items():
        scope_name = {'webpage': '网页', 'blog': '博客', 'scholar': '学术'}.get(scope, scope)
        print(f"  {scope_name}: {count} 条")
    
    return all_results

print("✓ 多主题搜索函数已定义（支持多资源类型和时间过滤）")

✓ 多主题搜索函数已定义（支持多资源类型和时间过滤）


## 步骤6: 执行搜索获取最新资讯

使用META API搜索感兴趣的技术资讯

In [ ]:
# 执行搜索
if META_API_KEY:
    news_data = search_multiple_topics(
        fields=TECH_FIELDS,
        focus_topics=FOCUS_TOPICS,
        api_key=META_API_KEY,
        scopes=SEARCH_SCOPES,  # 使用定义的资源类型
        time_filter=TIME_FILTER,  # 使用定义的时间过滤
        results_per_topic=3  # 每个主题获取3条结果
    )
    
    if news_data:
        print(f"\n" + "="*80)
        print(f"搜索完成统计:")
        print(f"  - 搜索时间: {news_data['search_date']}")
        print(f"  - 新闻条目数: {news_data['total_results']}")
        print(f"  - 资源类型: {', '.join(news_data['scopes'])}")
        print(f"  - 时间过滤: {news_data['time_filter'] if news_data['time_filter'] else '不限'}")
        print("="*80)
else:
    print("✗ 缺少API密钥，无法执行搜索")
    news_data = None


总共需要执行 18 次搜索
  - 查询主题: 6 个
  - 资源类型: 3 种 (webpage, blog, scholar)
  - 时间过滤: day


正在搜索资源类型: 网页

[1/18] 主题: 自动驾驶
正在搜索 [网页(最近1天)]: 自动驾驶...
✓ 搜索成功
{'credits': 3, 'searchParameters': {'q': '自动驾驶 最近1天', 'scope': 'webpage', 'size': 3, 'includeSummary': False, 'includeRawContent': False, 'conciseSnippet': False, 'format': 'chat_completions'}, 'webpages': [{'title': '特斯拉奥斯汀 Robotaxi 自动驾驶出租车再添 3 起事故，累计达 7 起', 'link': 'https://finance.sina.com.cn/tech/digi/2025-11-18/doc-infxviex4554462.shtml', 'score': 'medium', 'snippet': '与其他向 NHTSA 提交报告的企业不同，特斯拉对每起事故报告中的“事件描述”（narrative）部分均进行了删减，致使公众无从获知事故具体经过及责任归属。\n根据特斯拉报告中有限的披露信息可知：最新上报的三起事故中，一起为 Robotaxi 撞上一辆正在倒车的车辆，另一起涉及一名骑行者，第三起则与一只身份不明的动物有关。\n!\n[新浪科技公众号 新浪科技公众号](https://n.sinaimg.cn/tech/content/tech_qr2x.png)\n新浪科技公众号\n                    !\n[](//n.sinaimg.cn/tech/content/tech_weixin2.png)\n“掌”握科技鲜闻 \n!\n[](https://n.sinaimg.cn/tech/content/tech_weixin2.png)', 'position': 1, 'date': '2025年11月18日'}, {'title': '在无人驾驶出行这一赛道上，百度旗下的萝卜快跑再次交出了一份令人瞩目的成绩单'

## 步骤7: 可视化展示资讯摘要

以友好的格式展示获取到的技术资讯

In [ ]:
def display_news_summary(news_data: Dict[str, Any]):
    """
    以友好格式显示新闻摘要
    """
    if not news_data or not news_data.get('news_items'):
        print("✗ 无法显示：数据无效或为空")
        return
    
    # 显示标题
    print("\n" + "="*80)
    print(f"{'技术动态简报 (META搜索)':^76}")
    print(f"{'日期: ' + news_data.get('search_date', 'N/A'):^76}")
    print("="*80)
    
    # 显示关注领域
    fields = news_data.get('fields', [])
    topics = news_data.get('focus_topics', [])
    scopes = news_data.get('scopes', [])
    time_filter = news_data.get('time_filter', None)
    
    scope_names = [{'webpage': '网页', 'blog': '博客', 'scholar': '学术'}.get(s, s) for s in scopes]
    time_desc = {'day': '最近1天', 'week': '最近7天', 'month': '最近30天', None: '不限'}.get(time_filter, '不限')
    
    print(f"\n📊 关注领域: {', '.join(fields)}")
    print(f"🎯 重点主题: {', '.join(topics)}")
    print(f"📚 资源类型: {', '.join(scope_names)}")
    print(f"⏰ 时间范围: {time_desc}")
    print("\n" + "-"*80)
    
    # 显示新闻条目
    news_items = news_data.get('news_items', [])
    print(f"\n📰 共找到 {len(news_items)} 条技术资讯:\n")
    
    for i, item in enumerate(news_items, 1):
        # 相关性分数标记
        score = item.get('score', 'N/A')
        score_icon = {'high': '🔴', 'medium': '🟡', 'low': '🟢'}.get(score, '⚪')
        
        # 资源类型图标
        scope = item.get('scope', 'webpage')
        scope_icon = {'webpage': '🌐', 'blog': '📝', 'scholar': '🎓'}.get(scope, '📄')
        scope_name = item.get('scope_name', 'N/A')
        
        print(f"\n{i}. {score_icon} {scope_icon} [{item.get('category', 'N/A')}] {item.get('title', 'N/A')}")
        
        # 内容摘要
        content = item.get('content', 'N/A')
        # 清理内容中的多余空行和特殊字符
        content = re.sub(r'\n+', ' ', content)
        content = content.strip()
        
        print(f"\n   💬 内容摘要:")
        # 如果内容过长，截取前300字符
        if len(content) > 300:
            content = content[:300] + "..."
        print(f"   {content}")
        
        # 元信息
        print(f"\n   🔗 链接: {item.get('source_link', 'N/A')}")
        print(f"   📅 日期: {item.get('date', 'N/A')}")
        print(f"   📚 资源类型: {scope_name}")
        print(f"   ⭐ 相关性: {score}")
        print(f"   📍 排名位置: {item.get('position', 'N/A')}")
        
        print("\n" + "-"*80)

# 显示新闻摘要
if news_data:
    display_news_summary(news_data)


                              技术动态简报 (META搜索)                               
                          日期: 2025-11-10 11:02:40                           

📊 关注领域: 自动驾驶, 具身智能
🎯 重点主题: 特斯拉FSD, 人形机器人
📚 资源类型: 网页, 博客, 学术
⏰ 时间范围: 最近1天

--------------------------------------------------------------------------------

📰 共找到 24 条技术资讯:


1. 🟡 🌐 [自动驾驶] 精品：2009年高考语文寒假作业最新设计

   💬 内容摘要:
   邮晚蒂纠纽 障秧叮仗藐爱讹工乱骚吼 格痕刁灵佑峻犁冕衷晋省 茬乎谦叼烫熄狐乓应颁油 媳廖脐屁途反弦靖豢丧氧 防滦凳屎曾促蜘陋屏悔臀 应发遁镶仇婶砖唉涩捞亢 巢乒喜辖蔚躲虑棘孩枚液 吱茨诚税饭弛症脉邑蚤踩 毯韶窝移职李肝镀郴荣磅 瑞录事鄂绩吩攫托捅茧圾 樊淆翁能锻款孰邓蚁恭依 揉淳首派欢柔删尼柔苦藻 涤幅净柬哦瞅墨宜削虫钒 耕讹毡诱时栗咆村腹赐伺 汝吓携油潜讫嫉嫌疆政契 鳞呛涧填赦蔑久闭炬食欠 接乎蔗拯裴雁奖姓杨舞起 奈胰藐铣铰嫉亥浊耽赞 峨稗瘪戊燥李讼彪币樊仿 就楷澡破髓阴虽么寇褒幌 瘴抨按抠宴渺烈忽筑剐颠 株锌碱翟党柴坯乾吐癌斋 栗胡滞耍侮竟溺邹又潞 www.dearedu. com 苫拓卓...

   🔗 链接: https://www.doc88.com/p-652201074679.html
   📅 日期: 2009年
   📚 资源类型: 网页
   ⭐ 相关性: medium
   📍 排名位置: 1

--------------------------------------------------------------------------------

2. 🟡 🌐 [自动驾驶] 四级答题技巧

   💬 内容摘要:
   关键来了！！！ 要是写不出来那些词， 但是你知道意思的， 可以把整句话的意思给替换了， 不一定用原来的句子！ 写完之后，一定要检查时态， 单复数。 写这 3 个句子， 越简

## 步骤8: 保存数据到文件

将获取的资讯保存为JSON文件，方便后续分析和使用

In [ ]:
def save_news_to_file(news_data: Dict[str, Any], filename: str = None) -> str:
    """
    保存新闻数据到JSON文件
    
    参数:
        news_data: 新闻数据字典
        filename: 文件名（可选），默认使用时间戳命名
    
    返回:
        保存的文件路径
    """
    if not news_data or not news_data.get('news_items'):
        print("✗ 数据无效，无法保存")
        return None
    
    # 生成文件名
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"tech_news_meta_{timestamp}.json"
    
    try:
        # 保存JSON文件
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(news_data, f, ensure_ascii=False, indent=2)
        
        file_size = os.path.getsize(filename)
        print(f"✓ 数据已保存到: {filename}")
        print(f"  文件大小: {file_size} 字节")
        
        return filename
    except Exception as e:
        print(f"✗ 保存文件失败: {e}")
        return None

# 保存数据
if news_data:
    saved_file = save_news_to_file(news_data)
    if saved_file:
        print(f"\n可以使用以下代码重新加载数据:")
        print(f"with open('{saved_file}', 'r', encoding='utf-8') as f:")
        print(f"    news_data = json.load(f)")

✓ 数据已保存到: tech_news_meta_20251110_110255.json
  文件大小: 26830 字节

可以使用以下代码重新加载数据:
with open('tech_news_meta_20251110_110255.json', 'r', encoding='utf-8') as f:
    news_data = json.load(f)


## 步骤9: 数据分析和统计

对获取的资讯进行简单的统计分析

In [ ]:
def analyze_news_data(news_data: Dict[str, Any]):
    """
    分析新闻数据，生成统计报告
    """
    if not news_data or not news_data.get('news_items'):
        print("无法分析：数据无效")
        return
    
    news_items = news_data.get('news_items', [])
    
    print("\n" + "="*80)
    print(f"{'数据统计分析':^76}")
    print("="*80)
    
    # 1. 按类别(查询词)统计
    category_count = {}
    for item in news_items:
        category = item.get('category', '未分类')
        category_count[category] = category_count.get(category, 0) + 1
    
    print("\n📊 按搜索主题统计:")
    for category, count in sorted(category_count.items(), key=lambda x: x[1], reverse=True):
        print(f"   {category}: {count} 条")
    
    # 2. 按资源类型统计
    scope_count = {}
    for item in news_items:
        scope_name = item.get('scope_name', '未知')
        scope_count[scope_name] = scope_count.get(scope_name, 0) + 1
    
    print("\n📚 按资源类型统计:")
    for scope, count in sorted(scope_count.items(), key=lambda x: x[1], reverse=True):
        print(f"   {scope}: {count} 条")
    
    # 3. 按相关性得分统计
    score_count = {'high': 0, 'medium': 0, 'low': 0, 'N/A': 0}
    for item in news_items:
        score = item.get('score', 'N/A')
        if score in score_count:
            score_count[score] += 1
        else:
            score_count['N/A'] += 1
    
    print("\n⭐ 按相关性得分统计:")
    for score, count in score_count.items():
        if count > 0:
            score_name = {'high': '高', 'medium': '中', 'low': '低', 'N/A': '未知'}.get(score, score)
            print(f"   {score_name}: {count} 条")
    
    # 4. 按日期统计
    date_count = {}
    for item in news_items:
        date = item.get('date', '未知日期')
        if date != 'N/A' and date != '未知日期':
            date_count[date] = date_count.get(date, 0) + 1
    
    if date_count:
        print("\n📅 按日期统计 (Top 10):")
        top_dates = sorted(date_count.items(), key=lambda x: x[1], reverse=True)[:10]
        for date, count in top_dates:
            print(f"   {date}: {count} 条")
    
    # 5. 统计信息
    print("\n📈 总体统计:")
    print(f"   总条目数: {len(news_items)}")
    print(f"   搜索主题数: {len(category_count)}")
    print(f"   资源类型数: {len(scope_count)}")
    print(f"   高相关性结果: {score_count.get('high', 0)} 条")
    
    # 6. 链接域名统计
    domain_count = {}
    for item in news_items:
        link = item.get('source_link', '')
        if link and link != 'N/A':
            # 提取域名
            try:
                domain = re.search(r'https?://([^/]+)', link)
                if domain:
                    domain_name = domain.group(1)
                    domain_count[domain_name] = domain_count.get(domain_name, 0) + 1
            except:
                pass
    
    if domain_count:
        print("\n🌐 热门信息源 (Top 10):")
        top_domains = sorted(domain_count.items(), key=lambda x: x[1], reverse=True)[:10]
        for domain, count in top_domains:
            print(f"   {domain}: {count} 条")
    
    # 7. 交叉分析：按资源类型和相关性
    print("\n📊 资源类型 × 相关性交叉分析:")
    for scope in set([item.get('scope_name', '未知') for item in news_items]):
        scope_items = [item for item in news_items if item.get('scope_name') == scope]
        high_count = sum(1 for item in scope_items if item.get('score') == 'high')
        print(f"   {scope}: 共{len(scope_items)}条, 其中高相关{high_count}条")
    
    print("\n" + "="*80)

# 执行数据分析
if news_data:
    analyze_news_data(news_data)


                                   数据统计分析                                   

📊 按搜索主题统计:
   自动驾驶: 6 条
   具身智能: 6 条
   特斯拉FSD: 6 条
   人形机器人: 6 条

📚 按资源类型统计:
   网页: 12 条
   博客: 12 条

⭐ 按相关性得分统计:
   高: 8 条
   中: 16 条

📅 按日期统计 (Top 10):
   2009年: 2 条
   2015年08月01日: 2 条
   2019年02月04日: 2 条
   2025年05月13日: 2 条
   2025年07月13日: 2 条
   2025年02月06日: 2 条
   2022年08月16日: 2 条
   2024年05月11日: 2 条
   2025年03月26日: 2 条
   2024年08月09日: 2 条

📈 总体统计:
   总条目数: 24
   搜索主题数: 4
   资源类型数: 2
   高相关性结果: 8 条

🌐 热门信息源 (Top 10):
   finance.sina.com.cn: 10 条
   www.doc88.com: 6 条
   xueqiu.com: 4 条
   finance.sina.cn: 2 条
   new.qq.com: 2 条

📊 资源类型 × 相关性交叉分析:
   网页: 共12条, 其中高相关4条
   博客: 共12条, 其中高相关4条



## 步骤10: 完整流程封装

将以上步骤封装成一个完整的函数，方便重复使用

In [ ]:
def acquire_latest_news(fields: List[str], focus_topics: List[str], 
                       api_key: str, 
                       scopes: List[str] = ['webpage'],
                       time_filter: str = None,
                       results_per_topic: int = 3,
                       save_to_file: bool = True, 
                       show_analysis: bool = True) -> Dict[str, Any]:
    """
    完整的新闻获取流程（支持多资源类型和时间过滤）
    
    参数:
        fields: 技术领域列表
        focus_topics: 重点关注主题列表
        api_key: META API密钥
        scopes: 搜索范围列表，如 ['webpage', 'blog', 'scholar']
        time_filter: 时间过滤，可选 'day' (最近1天), 'week' (最近7天), 'month' (最近30天), None (不限)
        results_per_topic: 每个主题获取的结果数
        save_to_file: 是否保存到文件
        show_analysis: 是否显示统计分析
    
    返回:
        新闻数据字典
    """
    print("\n" + "="*80)
    print(f"{'开始获取最新技术资讯 (秘塔META)':^76}")
    print("="*80)
    
    # 1. 执行搜索
    print("\n[1/4] 执行多主题、多资源类型搜索...")
    news_data = search_multiple_topics(
        fields=fields,
        focus_topics=focus_topics,
        api_key=api_key,
        scopes=scopes,
        time_filter=time_filter,
        results_per_topic=results_per_topic
    )
    
    if not news_data or not news_data.get('news_items'):
        print("\n✗ 流程终止：搜索失败或无结果")
        return None
    
    # 2. 显示结果
    print("\n[2/4] 显示新闻摘要...")
    display_news_summary(news_data)
    
    # 3. 保存文件
    if save_to_file:
        print("\n[3/4] 保存数据到文件...")
        save_news_to_file(news_data)
    
    # 4. 统计分析（可选）
    if show_analysis:
        print("\n[4/4] 执行数据分析...")
        analyze_news_data(news_data)
    
    print("\n" + "="*80)
    print(f"{'流程完成！':^76}")
    print("="*80)
    
    return news_data

print("✓ 完整流程函数已定义（支持多资源类型和时间过滤）")
print("\n可以使用以下代码快速获取资讯:")
print("""
# 示例1: 获取最近1天的网页、博客、学术资讯
news_result = acquire_latest_news(
    fields=["人工智能", "机器学习"],
    focus_topics=["GPT", "Transformer"],
    api_key=META_API_KEY,
    scopes=['webpage', 'blog', 'scholar'],
    time_filter='day',  # 最近1天
    results_per_topic=3,
    save_to_file=True,
    show_analysis=True
)

# 示例2: 获取最近1周的网页资讯
news_result = acquire_latest_news(
    fields=["自动驾驶", "具身智能"],
    focus_topics=["特斯拉FSD", "人形机器人"],
    api_key=META_API_KEY,
    scopes=['webpage'],
    time_filter='week',  # 最近1周
    results_per_topic=5,
    save_to_file=True,
    show_analysis=True
)
""")

✓ 完整流程函数已定义（支持多资源类型和时间过滤）

可以使用以下代码快速获取资讯:

# 示例1: 获取最近1天的网页、博客、学术资讯
news_result = acquire_latest_news(
    fields=["人工智能", "机器学习"],
    focus_topics=["GPT", "Transformer"],
    api_key=META_API_KEY,
    scopes=['webpage', 'blog', 'scholar'],
    time_filter='day',  # 最近1天
    results_per_topic=3,
    save_to_file=True,
    show_analysis=True
)

# 示例2: 获取最近1周的网页资讯
news_result = acquire_latest_news(
    fields=["自动驾驶", "具身智能"],
    focus_topics=["特斯拉FSD", "人形机器人"],
    api_key=META_API_KEY,
    scopes=['webpage'],
    time_filter='week',  # 最近1周
    results_per_topic=5,
    save_to_file=True,
    show_analysis=True
)



## 步骤11: 高级功能 - 按时间过滤最新资讯

添加时间过滤功能，只保留最近的资讯

In [ ]:
def filter_by_date(news_data: Dict[str, Any], days_ago: int = 7) -> Dict[str, Any]:
    """
    按日期过滤新闻，只保留最近N天的资讯
    
    参数:
        news_data: 新闻数据
        days_ago: 保留最近几天的资讯
    
    返回:
        过滤后的新闻数据
    """
    if not news_data or not news_data.get('news_items'):
        return news_data
    
    from datetime import datetime, timedelta
    
    # 计算截止日期
    cutoff_date = datetime.now() - timedelta(days=days_ago)
    
    filtered_items = []
    for item in news_data['news_items']:
        date_str = item.get('date', '')
        if date_str and date_str != 'N/A':
            try:
                # 尝试解析多种日期格式
                # 例如: "2023年02月09日", "2023-02-09"
                date_str_cleaned = re.sub(r'[年月]', '-', date_str).replace('日', '')
                item_date = datetime.strptime(date_str_cleaned, "%Y-%m-%d")
                
                if item_date >= cutoff_date:
                    filtered_items.append(item)
            except:
                # 如果日期解析失败，保留该条目
                filtered_items.append(item)
        else:
            # 没有日期信息的也保留
            filtered_items.append(item)
    
    news_data['news_items'] = filtered_items
    news_data['total_results'] = len(filtered_items)
    
    print(f"✓ 过滤完成: 保留最近{days_ago}天的 {len(filtered_items)} 条资讯")
    
    return news_data

print("✓ 日期过滤函数已定义")

✓ 日期过滤函数已定义


## 总结

本笔记本演示了如何使用秘塔META API获取最新技术资讯，包括：

1. ✅ 环境配置和API密钥加载
2. ✅ 自定义技术领域和关注主题
3. ✅ 多资源类型搜索（网页、博客、学术）
4. ✅ 时间范围过滤（最近1天、7天、30天）
5. ✅ 调用META搜索API
6. ✅ 多主题批量搜索
7. ✅ 可视化展示资讯
8. ✅ 保存数据到文件
9. ✅ 数据统计分析
10. ✅ 完整流程封装

### 核心功能亮点

#### 1. 多资源类型支持
- **网页 (webpage)**: 普通网页内容，覆盖面最广
- **博客 (blog)**: 专注于技术博客和个人分享
- **学术 (scholar)**: 学术论文和研究成果

#### 2. 灵活的时间过滤
- **day**: 最近1天的资讯
- **week**: 最近7天的资讯  
- **month**: 最近30天的资讯
- **None**: 不限时间

#### 3. 详细的统计分析
- 按搜索主题统计
- 按资源类型统计
- 按相关性得分统计
- 按日期统计
- 热门信息源分析
- 资源类型×相关性交叉分析

### 使用建议

```python
# 基础用法：获取最近1天的多资源类型资讯
news_data = acquire_latest_news(
    fields=["自动驾驶", "具身智能", "大模型", "人工智能"],
    focus_topics=["特斯拉FSD", "人形机器人", "AI大模型突破"],
    api_key=META_API_KEY,
    scopes=['webpage', 'blog', 'scholar'],  # 三种资源类型
    time_filter='day',  # 最近1天
    results_per_topic=3,
    save_to_file=True,
    show_analysis=True
)

# 高级用法：自定义配置
TECH_FIELDS = ["量子计算", "区块链", "云计算"]
FOCUS_TOPICS = ["量子芯片", "DeFi", "边缘计算"]
SEARCH_SCOPES = ['webpage', 'scholar']  # 只搜索网页和学术
TIME_FILTER = 'week'  # 最近一周

news_data = search_multiple_topics(
    fields=TECH_FIELDS,
    focus_topics=FOCUS_TOPICS,
    api_key=META_API_KEY,
    scopes=SEARCH_SCOPES,
    time_filter=TIME_FILTER,
    results_per_topic=5
)
```

### META API vs HUNYUAN API 对比

| 特性 | META API | HUNYUAN API |
|------|----------|-------------|
| 搜索方式 | 直接搜索引擎 | AI联网搜索 |
| 结果格式 | 网页列表 | AI生成摘要 |
| 资源类型 | 网页/博客/学术 | 通用内容 |
| 时间过滤 | 支持（day/week/month） | 依赖AI理解 |
| 实时性 | 较好 | 依赖AI处理 |
| 结构化 | 搜索结果 | AI整理后数据 |
| 适用场景 | 快速获取多源信息 | 需要AI分析总结 |
| 覆盖面 | 更广（专业博客、学术） | AI筛选后的精选内容 |

### 配置说明

在步骤3中可以配置以下参数：

```python
# 定义搜索资源范围
SEARCH_SCOPES = ['webpage', 'blog', 'scholar']  # 可选任意组合

# 定义时间范围过滤
TIME_FILTER = 'day'   # 'day' / 'week' / 'month' / None
```

### 实用技巧

1. **快速浏览最新动态**：设置 `time_filter='day'` 和 `scopes=['webpage', 'blog']`
2. **深度研究**：设置 `scopes=['scholar']` 专注学术内容
3. **综合调研**：设置 `scopes=['webpage', 'blog', 'scholar']` 获取全面信息
4. **控制结果数量**：调整 `results_per_topic` 参数，建议 3-5 条

### 下一步

- ✅ 可以添加定时任务，定期获取资讯
- ✅ 可以集成更多数据分析和可视化功能（如词云、趋势图）
- ✅ 可以添加资讯去重和相似度分析
- ✅ 可以结合NLP技术进行关键信息提取
- ✅ 可以构建知识图谱，追踪技术发展趋势
- ✅ 可以导出为PDF报告或邮件推送

### 注意事项

1. API调用频率可能有限制，建议合理设置搜索数量
2. 时间过滤通过查询词实现，实际效果取决于搜索引擎的理解
3. 学术资源 (scholar) 更新可能相对较慢
4. 建议定期保存数据，便于历史对比和趋势分析